# Credit Card Fraud Detection (W16)

This week we will work with the widely used [Credit Card Fraud Detection dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud), originally released by the [Machine Learning Group at ULB](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud/versions/1/data). The `creditcard.csv.gz` file contains 284,807 transactions made by European cardholders over a set time period in September 2013. Only a few transactions are labeled as fraud, leaving the vast majority non-fraudulent. That large imbalance is very much the theme of this workshop.

The dataset was collected and analyzed as part of a collaboration between [Worldline](https://worldline.com/) and the [Machine Learning Group at ULB](https://mlg.ulb.ac.be/about/), a machine-learning research unit at Université libre de Bruxelles. It has since become one of the most widely used public fraud-detection datasets, and several research papers from that collaboration use it to study practical issues such as class imbalance, sampling, calibration, and realistic model evaluation.

Each row in the data corresponds to a credit card transaction. Much of the transaction information has been made anonymous so that the data could be published. A few notes about the data columns:

- `Class` is the target variable. `1` means fraud and `0` means non-fraud.
- `Time` is the number of seconds elapsed between a transaction and the first transaction in the dataset. It is **not** a wall-clock timestamp.
- `Amount` is the transaction amount.
- `V1` to `V28` are anonymized features obtained with [PCA](https://en.wikipedia.org/wiki/Principal_component_analysis). Because of confidentiality, the original feature meanings are not available.

Below are some suggested directions. This notebook is designed to tell a story: first understand the fraud problem, then build and evaluate models honestly, and finally turn model scores into a decision rule.

##### **Beginner**

In a binary classification problem each observation belongs to one of two classes. Here, the two classes are `fraud` and `non-fraud`. This dataset is also highly imbalanced, which means that there are many more instances of one class than there are of the other (here fraud is the rare occurrence). In this section, the goal is to get to know these ideas through the data before training any models.

- If you are totally new to Pandas, we recommend looking through the [Pandas getting started tutorial](https://pandas.pydata.org/docs/getting_started/index.html) before proceeding.
- Get started by orienting yourself with calls such as `.info()` and `.describe()`. Which column is the target, and why does that make this a binary classification problem?
- Compute `df["Class"].value_counts()` and `df["Class"].value_counts(normalize=True)`. How many fraudulent transactions are there, and what is the rate of fraud? This is what people mean when they say the dataset has class imbalance.
- Imagine a model that always predicts `0` (non-fraud). What accuracy would it get? Why does that number sound impressive at first, and why is it actually misleading here? Think it through.
- Compare the distribution of `Amount` for fraud and non-fraud transactions. A raw histogram is useful, but so is plotting `np.log1p(df["Amount"])` because the amount distribution is very skewed. Useful tools: [`hist`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.hist.html), [`np.log1p`](https://numpy.org/doc/stable/reference/generated/numpy.log1p.html).
- `Time` is measured in seconds since the first transaction in the dataset, not as a normal clock timestamp. Create an `Hour` feature such as `((df["Time"] // 3600) % 24).astype(int)`, then use [`groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) to plot both transaction counts and fraud rate by hour. Are fraudulent transactions more common at some times of day than others? Can we be sure that this `Hour` feature correctly corresponds to the time of the transactions?

##### **Intermediate**

This is where the modeling part begins. If you skipped the Beginner section, it is a good idea to first do some exploratory data analysis (specifically on class balance) so you know what kind of problem you are working with.

- Define `y = df["Class"]` and `X = df.drop(columns="Class")`. Then create a reproducible train/test split using [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html). If you are new to modeling, the idea is to keep some data unseen by the model when training (conventionally called the "test data"), in order to attain an unbiased estimate of the model's performance by evaluating it on that held-out data.
- A `stratified` split means that the class proportions are kept approximately the same in both the training and test sets. Create one split without `stratify=y` and one with `stratify=y`, then compare the fraud rates in the train and test sets. Try a few different `random_state` values. Does stratification matter much here? If the difference looks small, why might that be?
- Fit a [`DummyClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html) as an honest baseline. Evaluate it using accuracy, [`precision`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html), [`recall`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html), and a [`confusion matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html). If you are unfamiliar with these concepts, spend a couple of minutes trying to understand their definitions before you start. Also, what do you expect precision and recall to look like?
- Now fit a [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) model and compare it with the dummy baseline. If you feel comfortable, use a [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) with [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html). If these things are new to you, feel free to spend a couple of minutes reading up on them.
- Having good accuracy is not enough here, so move to metrics that are more informative for rare-event detection. Plot a [`precision-recall curve`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_curve.html) and compute the [`average precision score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html). Why are these more useful than accuracy in this problem?
- When one class is very rare, a model can end up paying too little attention to it. One family of methods for handling this is called *resampling*: in *downsampling* you remove many majority-class examples from the training data, and in *upsampling* you duplicate or resample minority-class examples so that they appear more often. If these terms are new to you, first read about [oversampling and undersampling](https://en.wikipedia.org/wiki/Oversampling_and_undersampling_in_data_analysis). Then implement both strategies for logistic regression, using tools such as [`DataFrame.sample`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html) or [`sklearn.utils.resample`](https://scikit-learn.org/stable/modules/generated/sklearn.utils.resample.html). Evaluate each model on the original test set and discuss what you gain and what you lose with each resampling method.

##### **Advanced**

On problems like this, there are several ways to move beyond a first baseline model. One can try more advanced models, give fraud cases more influence during training through weighting, and think more carefully about how predicted probabilities should be turned into actual decisions through threshold choice. If you start here, first make sure you understand the dataset through some exploratory data analysis and have trained at least one baseline model.

- Let's begin by trying a more flexible model family. Tree-based methods such as [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) and [`HistGradientBoostingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html) can capture nonlinear effects and interactions that simpler models cannot. Try at least one of them, compare it with a simpler model using the same evaluation setup as above, and see whether the added flexibility improves performance. If you are comfortable installing extra libraries, you might also consider [XGBoost](https://xgboost.readthedocs.io/en/stable/) or [LightGBM](https://lightgbm.readthedocs.io/en/latest/), which are also common choices for tabular data.
- One way to address class imbalance is to resample the training data, which was explored in the Intermediate section. Another is to leave the data as it is and instead weight the loss function, moving from something like $\frac{1}{n}\sum_{i=1}^n L(y_i, f(x_i))$ to $\frac{1}{n}\sum_{i=1}^n w_i L(y_i, f(x_i))$, where fraud examples are assigned larger weights. Since many estimators support this directly through `class_weight` or `sample_weight`, figure out how to do this with your model and compare the weighted and unweighted results. Compare this with the resampling approach and think through when each is preferable.
- For many classifiers, the most useful output is not the class label itself but the predicted probability of fraud. The final yes/no decision is usually made afterward by applying a threshold to that probability: for example, one could predict fraud whenever the model assigns probability above 0.5. But in a real application, that threshold should usually depend on the decision problem rather than on convention. To make this concrete, imagine the following scenario: a fraud-review team tells you that, over a two-day period, they can manually inspect at most about 1,400 flagged transactions, which is roughly 0.5% of all transactions in this dataset. If your test split contains 20% of the data, that corresponds to about 285 test-set transactions. They would also like at least 20% of reviewed transactions to actually be fraud if possible. Use predicted probabilities on the test set to search over thresholds, and choose one that respects the review budget while getting as close as possible to the 20% precision target and maximizing recall. Report the chosen threshold, the fraction of transactions flagged, the precision, the recall, and the confusion matrix. If no threshold satisfies both goals, explain which goal you would relax and why.

### Helpful references

- [Pandas getting started](https://pandas.pydata.org/docs/getting_started/index.html)
- [Pandas groupby user guide](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [Scikit-learn train/test splitting](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation)
- [Scikit-learn classification metrics guide](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics)
- [Scikit-learn precision-recall example](https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html)
- [Scikit-learn pipeline user guide](https://scikit-learn.org/stable/modules/compose.html#pipeline)


## Code


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("creditcard.csv.gz")
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
